# Data Cleaning & Processing

### Imports

In [ ]:
import pandas as pd
import numpy as np
import altair as alt
from IPython.display import display

### Data Viz Color Theme

In [ ]:
alt.data_transformers.disable_max_rows()

# Palette
ORANGE, NAVY, BLUE, GRAY, LIGHT = "#fc5d26", "#1f3864", "#2e75b6", "#e7e6e6", "#f2f2f2"

CHART_W, CHART_H = 730, 400

@alt.theme.register("precision", enable=True)
def precision_theme() -> alt.theme.ThemeConfig:
    return {
        "config": {
            "background": "white",
            "view": {"stroke": None},
            "font": "Arial",
            "title": {"font": "Arial", "fontSize": 18, "fontWeight": "bold",
                      "color": NAVY, "anchor": "start", "offset": 12},
            "axis": {"labelFont": "Arial", "titleFont": "Arial",
                     "labelFontSize": 12, "titleFontSize": 13,
                     "labelColor": "#595959", "titleColor": "#595959",
                     "gridColor": GRAY, "domainColor": GRAY, "tickColor": GRAY},
            "legend": {"labelFont": "Arial", "titleFont": "Arial",
                       "labelFontSize": 12, "titleFontSize": 13, "orient": "top"},
            "range": {
                "category": [NAVY, BLUE, ORANGE, "#7f7f7f", "#9dc3e6"],
                "ordinal": [LIGHT, "#9dc3e6", BLUE, NAVY],
                "ramp": [LIGHT, BLUE, NAVY],
            },
            "bar": {"color": NAVY},
            "line": {"color": NAVY, "strokeWidth": 3},
            "point": {"color": NAVY, "filled": True},
        }
    }

### Load the Data

In [ ]:
df = pd.read_csv('data.csv')

### Overall Data Structure

In [ ]:
print(f"Rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

print("Column Information:")
df.info()

print("Sample Rows")
display(df.head(10))

print("Data Summary:")
display(df.describe())

### Missingness

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": df.isnull().mean() * 100,
    "data_type": df.dtypes.astype(str)
}).sort_values("missing_percentage", ascending=False)


print("Missing Value Counts:")
display(missing_summary[missing_summary["missing_count"] > 0])

missing_data = missing_summary[
    missing_summary["missing_count"] > 0
].sort_values("missing_percentage")

if not missing_data.empty:
    base = alt.Chart(missing_data).encode(
        y=alt.Y("column:N", sort="-x", title=None),
        x=alt.X("missing_percentage:Q", title="Missing values (%)"),
    )
    bars = base.mark_bar().encode(
        color=alt.condition(alt.datum.missing_percentage > 20,
                            alt.value(ORANGE), alt.value(NAVY))  # flag columns >20% missing
    )
    labels = base.mark_text(align="left", dx=4, color="#595959").encode(
        text=alt.Text("missing_percentage:Q", format=".1f")
    )
    (bars + labels).properties(
        title="[Takeaway: e.g., 3 columns are >20% missing]",
        width=CHART_W - 150, height=max(150, 30 * len(missing_data))
    ).display()
else:
    print("No missing values detected.")

### Duplicates

In [ ]:
duplicate_count = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_count}")
print(f"Duplicate percentage: {duplicate_count / len(df) * 100}%")

# Check for duplicate records
for col in df.columns:
    if df[col].nunique(dropna=False) == 1:
        print(f"Constant column detected: {col}")

### Column Names

In [ ]:
df_clean = df.copy()

# Standardize column names
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^\w]+", "_", regex=True)
    .str.strip("_")
)

# Remove exact duplicate records
df_clean = df_clean.drop_duplicates().copy()

print("Updated column names:")
print(df_clean.columns.tolist())

### String Cleaning

In [ ]:
# Remove leading/trailing whitespace from text columns
string_columns = df_clean.select_dtypes(
    include=["object", "string"]
).columns

for col in string_columns:
    df_clean[col] = df_clean[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

# Standardize common representations of missing values
missing_values = [
    "", "NA", "N/A", "n/a", "NULL", "null", "None", "none"
]

df_clean = df_clean.replace(missing_values, np.nan)

### Data Types

In [ ]:
print("Column Types:")
display(
    pd.DataFrame({
        "column": df_clean.columns,
        "dtype": df_clean.dtypes.astype(str).values,
        "unique_values": [
            df_clean[col].nunique() for col in df_clean.columns
        ]
    })
)

# Identify columns that may contain dates
for col in df_clean.select_dtypes(include="object").columns:
    if any(word in col.lower() for word in ["date", "time", "timestamp"]):
        print(f"Potential date column: {col}")

        # Inspect before converting
        print(df_clean[col].head())

### Outlier Detection

In [ ]:
numeric_columns = df_clean.select_dtypes(include=np.number).columns

outlier_summary = []

for col in numeric_columns:

    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = (
        (df_clean[col] < lower_bound) |
        (df_clean[col] > upper_bound)
    )

    outlier_summary.append({
        "column": col,
        "outlier_count": outliers.sum(),
        "outlier_percentage": outliers.mean() * 100,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound
    })

outlier_df = pd.DataFrame(outlier_summary)

display(
    outlier_df.sort_values(
        "outlier_percentage", ascending=False
    )
)

# IMPORTANT:
# Do not automatically remove outliers.
# Determine whether they represent errors or meaningful observations.

### Final Structure

In [ ]:
print(f"Original shape: {df.shape}")
print(f"Cleaned shape: {df_clean.shape}")
print(f"Rows removed: {len(df) - len(df_clean)}")

print("\nRemaining missing values:")
display(
    df_clean.isnull().sum()
    .sort_values(ascending=False)
    .loc[lambda x: x > 0]
)

print("\nFinal data types:")
display(df_clean.dtypes)

print("\nCLEANED DATASET:")
display(df_clean.head())

# Save cleaned data if needed
df_clean.to_csv("cleaned_data.csv", index=False)